## 2027 market-share forecasts

Notebook 3 continues each county's relative 2019–2024 pace through Model 2. That is a ranking scenario, not a 2027 share level. This notebook scores **2027 conditions** with the frozen four-term profile from notebooks 4–6, then adds each county's historical departure so Kaiser-specific gaps the socioeconomic model does not see stay in the forecast.

`share_2027 = profile(X_2027) + departure_carry`

Inputs come from `mart_cty_forecast_inputs`, kept separate from the historical `mart_cty_yr_*` tables. The seed uses **DOF 2027 population and unemployment**, **Covered California 2027** premiums and carrier counts, and **ACS 2024 holds** for uninsured / income / poverty, plus simple stress scenarios.

This is a screening forecast of share, not an enrollment headcount plan.


In [ ]:
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path
from IPython.display import HTML, Markdown, display

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
})

FOUR_FORMULA = (
    "kp_market_share ~ log_population + uninsured_rate + unemployment_rate "
    "+ benchmark_silver_premium"
)
PROFILE_TERMS = ["log_population", "uninsured_rate", "unemployment_rate", "benchmark_silver_premium"]
SOUTHERN = ["Imperial", "Kern", "Orange", "Riverside", "San Bernardino", "San Diego", "Ventura"]
FORECAST_YEAR = 2027
FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def short_county(name):
    return name.replace(" County", "")


con = duckdb.connect("../data/processed/analysis.duckdb", read_only=True)
market = con.sql("select * from main_marts.mart_cty_yr_market").df()
performance = con.sql("select * from main_marts.mart_cty_yr_performance").df()
socioeconomic = con.sql("select * from main_marts.mart_cty_yr_socioeconomic").df()
forecast_long = con.sql("select * from main_marts.mart_cty_forecast_inputs").df()
footprint = con.sql("""
    select
        year,
        county_fips,
        bool_and(coalesce(is_partial_county, false)) as partial_county
    from main_staging.stg_cms_service_areas
    where issuer_id = '40513'
      and service_area_id = 'CAS002'
    group by 1, 2
""").df()

panel = (
    market.merge(performance, on=["county_fips", "year"], how="left", suffixes=("", "_performance"))
    .merge(socioeconomic, on=["county_fips", "year"], how="left", suffixes=("", "_socio"))
    .merge(footprint, on=["county_fips", "year"], how="left")
)
panel["county"] = panel["county_name"].map(short_county)
panel["log_population"] = np.log(panel["population_total"])

offered = panel[panel["kp_offered"] & panel["year"].between(2016, 2024)].copy()
measure_cols = [
    "kp_market_share", "population_total", "uninsured_rate", "unemployment_rate", "benchmark_silver_premium",
]
county = offered.groupby(["county_fips", "county"], as_index=False)[measure_cols].mean()
county["log_population"] = np.log(county["population_total"])
sample = county.dropna(subset=measure_cols).copy()
sample["southern"] = sample["county"].isin(SOUTHERN).astype(int)

profile_model = smf.ols(FOUR_FORMULA, data=sample).fit(cov_type="HC1")
regional_model = smf.ols(FOUR_FORMULA + " + southern", data=sample).fit(cov_type="HC1")
residual_sd = float(np.sqrt(profile_model.mse_resid))
regional_sd = float(np.sqrt(regional_model.mse_resid))

departures = sample[["county_fips", "county", "kp_market_share", "southern"]].copy()
departures["expected_hist"] = profile_model.fittedvalues.to_numpy()
departures["departure"] = profile_model.resid.to_numpy()
departures["departure_with_region"] = regional_model.resid.to_numpy()
departures["abs_error_pp"] = departures["departure"].abs() * 100
departures["confidence"] = np.select(
    [
        departures["abs_error_pp"] <= 0.5 * residual_sd * 100,
        departures["abs_error_pp"] <= residual_sd * 100,
    ],
    ["Higher", "Moderate"],
    default="Lower",
)

year_2025 = panel.loc[panel["year"] == 2025, [
    "county_fips", "partial_county", "kp_offered", "kp_market_share", "total_marketplace_enrollment",
]].rename(columns={"kp_market_share": "share_2025", "total_marketplace_enrollment": "market_2025"})
departures = departures.merge(year_2025, on="county_fips", how="left")
departures["footprint"] = np.where(departures["partial_county"], "Some ZIP codes", "Whole county")
departures["region"] = np.where(departures["southern"].eq(1), "Southern California", "Northern California")

sources = (
    forecast_long.groupby(["scenario", "measure"], as_index=False)["source"]
    .agg(lambda s: ", ".join(sorted(set(s))))
)
display(Markdown(
    f"Frozen four-term profile on **{len(sample)}** counties "
    f"(adj. R² **{profile_model.rsquared_adj:.2f}**, residual SE **{residual_sd * 100:.1f}** pp). "
    f"Forecast mart has **{forecast_long['county_fips'].nunique()}** counties, "
    f"year **{sorted(forecast_long['forecast_year'].unique())}**, "
    f"scenarios **{sorted(forecast_long['scenario'].unique())}**."
))
display(Markdown("**Input provenance by scenario and measure**"))
display(HTML(sources.pivot(index="measure", columns="scenario", values="source").fillna("").to_html()))


## Build 2027 covariate panels by scenario

Each scenario is pivoted wide. Profile terms must be present. Los Angeles has no county benchmark premium in the bridge file, so it cannot be scored. Monterey is offered in 2025 but was outside the 2016–2024 estimation sample; it is scored as an **opening** without a sticky historical departure.


In [ ]:
REQUIRED = ["population_total", "uninsured_rate", "unemployment_rate", "benchmark_silver_premium"]
fc = forecast_long[forecast_long["forecast_year"].eq(FORECAST_YEAR)].copy()
wide = (
    fc.pivot_table(
        index=["county_fips", "county_name", "scenario"],
        columns="measure",
        values="value",
        aggfunc="first",
    )
    .reset_index()
)
wide.columns.name = None
wide["county"] = wide["county_name"].map(short_county)
wide["log_population"] = np.log(wide["population_total"])
wide["southern"] = wide["county"].isin(SOUTHERN).astype(int)
wide["is_monterey"] = wide["county"].eq("Monterey")
wide["is_los_angeles"] = wide["county"].eq("Los Angeles")

# Footprint / offered status from 2025
status_2025 = panel.loc[panel["year"] == 2025, ["county_fips", "kp_offered", "partial_county"]]
wide = wide.merge(status_2025, on="county_fips", how="left")
wide["footprint"] = np.where(wide["partial_county"], "Some ZIP codes", "Whole county")
wide["region"] = np.where(wide["southern"].eq(1), "Southern California", "Northern California")

scoreable = wide.dropna(subset=REQUIRED).copy()
skipped = wide.loc[~wide.index.isin(scoreable.index), ["county", "scenario"] + [c for c in REQUIRED if c in wide.columns]]
display(Markdown(
    f"**{scoreable['county_fips'].nunique()}** counties are scoreable in at least one scenario. "
    f"**Los Angeles** is skipped (no county premium). "
    f"**Monterey** is scored as an opening (no sticky departure from the 2016–2024 fit)."
))
if len(skipped):
    display(Markdown("Rows missing a required profile input (shown once per county-scenario):"))
    display(HTML(skipped.drop_duplicates("county").head(10).to_html(index=False)))


## Score: profile(X_2027) + sticky departure

The profile coefficients stay frozen. `departure_carry` is the county's 2016–2024 mean residual. Extreme departures are also shown with mild shrinkage toward zero (`departure * residual_SE² / (residual_SE² + departure²)`), which pulls giant outliers in without discarding the sticky Kaiser-specific gap. Monterey gets `departure_carry = 0` and an opening flag. Bands around the base path use ±1 residual SE.


In [ ]:
# Design matrices for predict
exog_cols = ["Intercept"] + PROFILE_TERMS
scoreable = scoreable.copy()
scoreable["Intercept"] = 1.0
params = profile_model.params.reindex(exog_cols)
scoreable["profile_share"] = scoreable[exog_cols].to_numpy() @ params.to_numpy()

# Regional companion
exog_reg = exog_cols + ["southern"]
params_reg = regional_model.params.reindex(exog_reg)
scoreable["profile_share_with_region"] = scoreable[exog_reg].to_numpy() @ params_reg.to_numpy()

carry = departures[["county_fips", "departure", "departure_with_region", "confidence", "share_2025", "market_2025"]].copy()
scoreable = scoreable.merge(carry, on="county_fips", how="left")
# Opening / unscored history: no sticky residual
scoreable["departure"] = scoreable["departure"].fillna(0.0)
scoreable["departure_with_region"] = scoreable["departure_with_region"].fillna(0.0)
scoreable["confidence"] = scoreable["confidence"].fillna("Opening / unscored")
# Mild empirical-Bayes style shrink for extreme residuals
se2 = residual_sd ** 2
scoreable["departure_shrunk"] = scoreable["departure"] * (
    se2 / (se2 + scoreable["departure"] ** 2)
)
scoreable.loc[scoreable["is_monterey"], "departure_shrunk"] = 0.0

scoreable["share_2027"] = scoreable["profile_share"] + scoreable["departure"]
scoreable["share_2027_shrunk"] = scoreable["profile_share"] + scoreable["departure_shrunk"]
scoreable["share_2027_with_region"] = scoreable["profile_share_with_region"] + scoreable["departure_with_region"]
scoreable["share_2027_low"] = scoreable["share_2027"] - residual_sd
scoreable["share_2027_high"] = scoreable["share_2027"] + residual_sd
# clip display to [0, 1] for share tables, keep uncapped internals for deltas
for col in ["share_2027", "share_2027_shrunk", "share_2027_with_region", "share_2027_low", "share_2027_high", "profile_share"]:
    scoreable[col + "_clip"] = scoreable[col].clip(0, 1)

base = scoreable[scoreable["scenario"].eq("base")].copy()
base = base.sort_values("share_2027", ascending=False).reset_index(drop=True)

base_view = pd.DataFrame({
    "county": base["county"],
    "flag": np.where(base["is_monterey"], "Opening", np.where(base["confidence"].eq("Lower"), "Lower confidence", "")),
    "region": base["region"],
    "footprint": base["footprint"],
    "share_2025": (base["share_2025"] * 100).map(lambda v: "" if pd.isna(v) else f"{v:.0f}%"),
    "profile_2027": (base["profile_share_clip"] * 100).map(lambda v: f"{v:.0f}%"),
    "departure_pp": (base["departure"] * 100).map(lambda v: f"{v:+.1f}"),
    "share_2027": (base["share_2027_clip"] * 100).map(lambda v: f"{v:.0f}%"),
    "band": base.apply(
        lambda r: f"{r.share_2027_low_clip * 100:.0f}–{r.share_2027_high_clip * 100:.0f}%",
        axis=1,
    ),
    "confidence": base["confidence"],
})
display(Markdown("**Base 2027 score** = frozen profile on bridge inputs + sticky historical departure. Band is ±1 residual SE around that path."))
display(HTML(base_view.to_html(index=False)))

display(Markdown(
    f"Median base 2027 share among scoreable offered counties is "
    f"**{base['share_2027_clip'].median() * 100:.0f}%** "
    f"(profile alone **{base['profile_share_clip'].median() * 100:.0f}%**; "
    f"median sticky departure **{base['departure'].median() * 100:+.1f}** pp). "
    f"Monterey opening score (profile only): "
    f"**{base.loc[base['is_monterey'], 'profile_share_clip'].mul(100).round(0).astype(int).tolist()}%**."
))


## Scenario bands

Stress scenarios move one input at a time from the bridge base: uninsured +1 pp, unemployment +1 pp, and benchmark premium +$50. The table shows the base sticky-departure path and the scenario range across those cases (plus base).


In [ ]:
scenario_order = ["base", "higher_uninsured", "softer_labor", "premium_up"]
scored = scoreable[scoreable["scenario"].isin(scenario_order)].copy()

pivot = scored.pivot_table(
    index=["county_fips", "county", "region", "footprint", "confidence", "is_monterey"],
    columns="scenario",
    values="share_2027_clip",
    aggfunc="first",
).reset_index()
pivot.columns.name = None
for scen in scenario_order:
    if scen not in pivot.columns:
        pivot[scen] = np.nan
pivot["scenario_low"] = pivot[scenario_order].min(axis=1)
pivot["scenario_high"] = pivot[scenario_order].max(axis=1)
pivot["scenario_span_pp"] = (pivot["scenario_high"] - pivot["scenario_low"]) * 100
pivot = pivot.sort_values("base", ascending=False)

scenario_view = pd.DataFrame({
    "county": pivot["county"],
    "base": (pivot["base"] * 100).map(lambda v: f"{v:.0f}%"),
    "higher_uninsured": (pivot["higher_uninsured"] * 100).map(lambda v: f"{v:.0f}%"),
    "softer_labor": (pivot["softer_labor"] * 100).map(lambda v: f"{v:.0f}%"),
    "premium_up": (pivot["premium_up"] * 100).map(lambda v: f"{v:.0f}%"),
    "scenario_range": pivot.apply(
        lambda r: f"{r.scenario_low * 100:.0f}–{r.scenario_high * 100:.0f}%",
        axis=1,
    ),
    "span_pp": pivot["scenario_span_pp"].map(lambda v: f"{v:.1f}"),
    "confidence": pivot["confidence"],
})
display(HTML(scenario_view.to_html(index=False)))

display(Markdown(
    f"Across counties, the median span across the four scenarios is "
    f"**{pivot['scenario_span_pp'].median():.1f}** percentage points. "
    f"That input uncertainty is much smaller than the model residual SE "
    f"(**{residual_sd * 100:.1f}** pp), so historical fit quality dominates the band."
))

# Chart: base forecast vs 2025 actual for counties with both
compare = base.dropna(subset=["share_2025"]).sort_values("share_2027")
fig, ax = plt.subplots(figsize=(8, 8.5))
y = np.arange(len(compare))
ax.hlines(y, compare["share_2025"] * 100, compare["share_2027_clip"] * 100, color="#d0d5dd", linewidth=1)
ax.scatter(compare["share_2025"] * 100, y, color="#98a2b3", s=28, label="2025 actual", zorder=3)
ax.scatter(compare["share_2027_clip"] * 100, y, color="#1f4e79", s=28, label="2027 base score", zorder=3)
ax.set_yticks(y)
ax.set_yticklabels(compare["county"], fontsize=8)
ax.set_xlabel("Kaiser market share (%)")
ax.set_title("2025 actual share vs 2027 base score (profile + sticky departure)")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "07_share_2025_vs_2027_base.png", dpi=150, bbox_inches="tight")
plt.show()


## Companion views: region term and shrunk departures

Two sensitivity columns sit beside the base sticky path: (1) profile with the Southern California indicator plus regional residual, and (2) profile plus mildly shrunk departure. Large gaps between these and the base path are places to treat the point forecast cautiously.


In [ ]:
sens = base[[
    "county", "region", "confidence", "is_monterey",
    "share_2027_clip", "share_2027_with_region_clip", "share_2027_shrunk_clip",
    "departure", "departure_shrunk",
]].copy()
sens_view = pd.DataFrame({
    "county": sens["county"],
    "base_sticky": (sens["share_2027_clip"] * 100).map(lambda v: f"{v:.0f}%"),
    "with_region": (sens["share_2027_with_region_clip"] * 100).map(lambda v: f"{v:.0f}%"),
    "shrunk_departure": (sens["share_2027_shrunk_clip"] * 100).map(lambda v: f"{v:.0f}%"),
    "departure_pp": (sens["departure"] * 100).map(lambda v: f"{v:+.1f}"),
    "shrunk_pp": (sens["departure_shrunk"] * 100).map(lambda v: f"{v:+.1f}"),
    "confidence": sens["confidence"],
    "region": sens["region"],
})
display(HTML(sens_view.sort_values("county").to_html(index=False)))

# Scenario effect sizes via profile coefficients (illustrative)
coef = profile_model.params
display(Markdown(
    "Approximate profile response to the packaged stresses "
    f"(holding other terms fixed): uninsured +1 pp → "
    f"**{coef['uninsured_rate'] * 100:+.1f}** share points; "
    f"unemployment +1 pp → **{coef['unemployment_rate'] * 100:+.1f}**; "
    f"benchmark +$50 → **{coef['benchmark_silver_premium'] * 50 * 100:+.1f}**. "
    "These are between-county associations, not within-county causal effects."
))


## Flags the Kaiser audience should keep next to the table

- **Los Angeles** is unscored: no single county benchmark premium.
- **Monterey** is an opening score (profile only). Compare its path with Santa Cruz from notebook 5 rather than treating the 2027 number as a mature-footprint forecast.
- **Lower confidence** tags come from notebook 6 absolute residuals beyond one residual SE. Use ops / network review before acting on those counties.
- **Southern California** still carries the large regional gap; the with-region companion is the fairer screen inside the South.
- **Uninsured rates** are still ACS 2024 holds. Population, unemployment, premiums, and carrier counts are official 2027 sources in the seed.
- Headline remains **share**, not members, unless a 2027 Covered California enrollment denominator is supplied separately.
- Higher 2027 Silver benchmarks lift many profile scores; read that as the historical price-position association, not a causal premium lever. Keep the ±1 residual-SE band next to any point estimate.

Notebook 3's relative-pace ranking remains useful for whether the ordering of counties drifts. This notebook answers a different question: given 2027 conditions, what share does the socioeconomic / market tool imply once sticky Kaiser-specific departures are carried forward.
